# Strong GNN Baseline Upgrade

Один notebook для подготовки данных, валидации, обучения финальной модели и генерации `submission.csv`.


In [1]:
import importlib.util
import os
import subprocess
import sys

import torch


def ensure_package(module_name: str, install_args: list[str]) -> None:
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *install_args])


torch_version = torch.__version__
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}.html'

if importlib.util.find_spec('torch_geometric') is None:
    ensure_package('torch_scatter', ['torch-scatter', '-f', wheel_url])
    ensure_package('torch_sparse', ['torch-sparse', '-f', wheel_url])
    ensure_package('torch_geometric', ['torch-geometric'])

ensure_package('pyarrow', ['pyarrow'])

print('Torch version:', torch.__version__)


Torch version: 2.10.0+cu128


In [2]:
import copy
import gc
import getpass
import math
import random
import shutil
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
import tqdm.auto as tqdm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import TransformerConv

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 300)

print('CUDA available:', torch.cuda.is_available())


CUDA available: True


In [3]:
CONFIG = {
    'seed': 402,
    'data_dir': './',
    'target_names': ['label_3', 'label_4', 'label_5', 'label_6'],
    'node_type_col': 'node_feature_1',
    'num_folds': 3,
    'hidden_size': 192,
    'input_mlp_size': 256,
    'heads': 4,
    'dropout': 0.2,
    'batch_size': 256,
    'num_neighbors': [-1, -1, -1],
    'max_epochs': 40,
    'early_stopping_patience': 5,
    'scheduler_patience': 2,
    'lr': 2e-3,
    'weight_decay': 1e-4,
    'grad_clip': 1.0,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'storage_dtype': 'float16',
    'run_cv': True,
    'run_final_fit': True,
    'save_oof': True,
}


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(CONFIG['seed'])
CONFIG


{'seed': 402,
 'data_dir': './',
 'target_names': ['label_3', 'label_4', 'label_5', 'label_6'],
 'node_type_col': 'node_feature_1',
 'num_folds': 3,
 'hidden_size': 192,
 'input_mlp_size': 256,
 'heads': 4,
 'dropout': 0.2,
 'batch_size': 256,
 'num_neighbors': [-1, -1, -1],
 'max_epochs': 40,
 'early_stopping_patience': 5,
 'scheduler_patience': 2,
 'lr': 0.002,
 'weight_decay': 0.0001,
 'grad_clip': 1.0,
 'device': 'cuda',
 'storage_dtype': 'float16',
 'run_cv': True,
 'run_final_fit': True,
 'save_oof': True}

## Data Prep

Поддерживаются два типовых режима:
- Kaggle competition input (`/kaggle/input/gnn-multitarget-2025`)
- локальная папка после распаковки архива (`./` или `./gnn-multitarget-2025`)


In [4]:
def resolve_data_dir(config: dict) -> Path:
    required = {'df_edges.parquet', 'df_nodes.parquet', 'train.csv', 'test.csv'}

    def has_required_files(candidate: Path) -> bool:
        return candidate.is_dir() and required.issubset({p.name for p in candidate.iterdir()})

    roots = [
        Path(config['data_dir']).expanduser(),
        Path.cwd(),
        Path('/kaggle/input/gnn-multitarget-2025'),
        Path('/kaggle/input'),
    ]

    checked = set()
    for root in roots:
        for candidate in [root, root / 'gnn-multitarget-2025']:
            candidate = candidate.resolve()
            if candidate in checked:
                continue
            checked.add(candidate)
            if has_required_files(candidate):
                return candidate

        if root.is_dir():
            for child in root.iterdir():
                if not child.is_dir():
                    continue
                candidate = child.resolve()
                if candidate in checked:
                    continue
                checked.add(candidate)
                if has_required_files(candidate):
                    return candidate

    raise FileNotFoundError(
        'Could not find a directory with df_edges.parquet, df_nodes.parquet, train.csv and test.csv. '
        f'Checked from cwd={Path.cwd()}'
    )


def ensure_kaggle_cli() -> None:
    if shutil.which('kaggle') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'])


def ensure_competition_data(config: dict) -> Path:
    try:
        return resolve_data_dir(config)
    except FileNotFoundError:
        pass

    ensure_kaggle_cli()

    if not os.environ.get('KAGGLE_API_TOKEN'):
        os.environ['KAGGLE_API_TOKEN'] = getpass.getpass('Enter Kaggle API token: ').strip()

    download_dir = Path(config['data_dir']).expanduser()
    if str(download_dir) in {'.', ''}:
        if 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_GPU' in os.environ:
            download_dir = Path('/content/gnn-multitarget-2025')
        else:
            download_dir = Path.cwd()

    download_dir.mkdir(parents=True, exist_ok=True)
    zip_path = download_dir / 'gnn-multitarget-2025.zip'

    if not zip_path.exists():
        subprocess.check_call(
            ['kaggle', 'competitions', 'download', '-c', 'gnn-multitarget-2025', '-p', str(download_dir)]
        )

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(download_dir)

    return resolve_data_dir({'data_dir': str(download_dir)})


def build_node_features_from_parquet(parquet_path: Path, node_type_col: str, storage_dtype: str, batch_size: int = 50_000):
    parquet_file = pq.ParquetFile(parquet_path)
    all_columns = parquet_file.schema_arrow.names
    feature_cols = [c for c in all_columns if c not in {'index', node_type_col}]
    num_rows = parquet_file.metadata.num_rows
    num_features = len(feature_cols)

    storage_np_dtype = np.float16 if storage_dtype == 'float16' else np.float32
    memmap_path = parquet_path.with_suffix(f'.{storage_dtype}.mmap')
    x_memmap = np.memmap(memmap_path, mode='w+', dtype=storage_np_dtype, shape=(num_rows, num_features))
    node_type = np.empty(num_rows, dtype=np.int64)
    accumulators = {}
    row_start = 0
    total_batches = math.ceil(num_rows / batch_size)

    for batch in tqdm.tqdm(
        parquet_file.iter_batches(batch_size=batch_size, columns=[node_type_col, *feature_cols], use_threads=True),
        total=total_batches,
        desc='Node parquet batches',
    ):
        batch_df = batch.to_pandas(split_blocks=True, self_destruct=True)
        batch_node_type = batch_df[node_type_col].to_numpy(dtype=np.int64, copy=False)
        batch_values = batch_df[feature_cols].to_numpy(dtype=np.float32, copy=False)
        row_end = row_start + len(batch_df)

        x_memmap[row_start:row_end] = batch_values.astype(storage_np_dtype, copy=False)
        node_type[row_start:row_end] = batch_node_type

        for group in np.unique(batch_node_type):
            group = int(group)
            group_mask = batch_node_type == group
            group_values = batch_values[group_mask]
            valid_mask = ~np.isnan(group_values)
            filled = np.nan_to_num(group_values, nan=0.0).astype(np.float64, copy=False)

            if group not in accumulators:
                accumulators[group] = {
                    'sum': np.zeros(num_features, dtype=np.float64),
                    'sumsq': np.zeros(num_features, dtype=np.float64),
                    'count': np.zeros(num_features, dtype=np.int64),
                }

            accumulators[group]['sum'] += filled.sum(axis=0)
            accumulators[group]['sumsq'] += np.square(filled).sum(axis=0)
            accumulators[group]['count'] += valid_mask.sum(axis=0)

        row_start = row_end
        del batch_df, batch_values, batch_node_type, filled, valid_mask
        gc.collect()

    x_memmap.flush()
    feature_stats = {}
    for group, acc in accumulators.items():
        active_idx = np.flatnonzero(acc['count'] > 0)
        means = np.zeros(len(active_idx), dtype=np.float32)
        stds = np.ones(len(active_idx), dtype=np.float32)
        if len(active_idx) > 0:
            means = (acc['sum'][active_idx] / acc['count'][active_idx]).astype(np.float32)
            variances = (acc['sumsq'][active_idx] / acc['count'][active_idx]) - np.square(means.astype(np.float64))
            stds = np.sqrt(np.clip(variances, 1e-6, None)).astype(np.float32)
        feature_stats[group] = {
            'feature_idx': active_idx.tolist(),
            'median': means.copy(),
            'mean': means,
            'std': stds,
            'feature_names': [feature_cols[idx] for idx in active_idx],
        }

    return torch.from_numpy(x_memmap), torch.from_numpy(node_type).long(), feature_cols, feature_stats, str(memmap_path)


def build_edge_features(df_edges: pd.DataFrame, storage_dtype: str):
    edge_index = torch.from_numpy(df_edges[['index1', 'index2']].to_numpy(dtype=np.int64).T)
    edge_feature_cols = [c for c in df_edges.columns if c not in {'index1', 'index2'}]
    edge_frame = df_edges[edge_feature_cols].astype(np.float32)
    means = edge_frame.mean(axis=0)
    stds = edge_frame.std(axis=0).replace(0, 1.0).fillna(1.0)
    edge_values = ((edge_frame - means) / stds).fillna(0.0).to_numpy(
        dtype=np.float16 if storage_dtype == 'float16' else np.float32
    )
    edge_attr = torch.from_numpy(edge_values)
    edge_stats = {
        'feature_cols': edge_feature_cols,
        'mean': means.to_numpy(dtype=np.float32),
        'std': stds.to_numpy(dtype=np.float32),
    }
    return edge_index, edge_attr, edge_stats


def build_targets(num_nodes: int, df_train: pd.DataFrame, target_names: list[str]):
    df_y = df_train.pivot(index='index', columns='label_type', values='y')
    df_y = df_y.reindex(np.arange(num_nodes))
    label_values = df_y[target_names].to_numpy(dtype=np.float32)
    y = torch.from_numpy(np.nan_to_num(label_values, nan=0.0).astype(np.float32))
    label_mask = torch.from_numpy(~np.isnan(label_values))
    has_target = label_mask.any(dim=1)
    return df_y, y, label_mask, has_target


def make_split_folds(df_y: pd.DataFrame, node_type: torch.Tensor, target_names: list[str], n_splits: int, seed: int):
    labeled_mask = df_y[target_names].notna().any(axis=1).to_numpy()
    labeled_nodes = np.flatnonzero(labeled_mask)
    labeled_df = df_y.iloc[labeled_nodes][target_names]
    observed = labeled_df.notna().to_numpy()
    values = labeled_df.fillna(-1).to_numpy()
    primary_labels = []

    for row_values, row_observed in zip(values, observed):
        pos_idx = np.flatnonzero(row_values == 1)
        if len(pos_idx) > 0:
            primary = int(pos_idx.max())
        else:
            primary = int(np.flatnonzero(row_observed)[0])
        primary_labels.append(primary)

    stratify_key = np.array([
        f"{int(t)}_{int(label)}"
        for t, label in zip(node_type.cpu().numpy()[labeled_nodes], primary_labels)
    ])

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []
    for fold_id, (train_idx, valid_idx) in enumerate(skf.split(labeled_nodes, stratify_key)):
        folds.append(
            {
                'fold_id': fold_id,
                'train_nodes': torch.from_numpy(labeled_nodes[train_idx]).long(),
                'valid_nodes': torch.from_numpy(labeled_nodes[valid_idx]).long(),
            }
        )

    return folds, torch.from_numpy(labeled_nodes).long()


In [5]:
data_dir = ensure_competition_data(CONFIG)
print('Using data directory:', data_dir)

df_train = pd.read_csv(data_dir / 'train.csv')
df_test = pd.read_csv(data_dir / 'test.csv')

print('df_train:', df_train.shape)
print('df_test:', df_test.shape)


Using data directory: /content/gnn-multitarget-2025
df_train: (10185, 4)
df_test: (5018, 3)


In [6]:
print('Streaming node table from parquet...')
node_parquet_path = data_dir / 'df_nodes.parquet'
num_nodes = pq.ParquetFile(node_parquet_path).metadata.num_rows
x, node_type, node_feature_cols, feature_stats, node_memmap_path = build_node_features_from_parquet(
    node_parquet_path,
    CONFIG['node_type_col'],
    CONFIG['storage_dtype'],
)

df_y, y, label_mask, has_target = build_targets(num_nodes, df_train, CONFIG['target_names'])
test_index = torch.from_numpy(df_test['index'].unique()).long()

print('Loading edge table...')
df_edges = pd.read_parquet(data_dir / 'df_edges.parquet')
num_edges = len(df_edges)
edge_index, edge_attr, edge_stats = build_edge_features(df_edges, CONFIG['storage_dtype'])
del df_edges
gc.collect()

data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr,
    y=y,
    label_mask=label_mask,
    has_target=has_target,
    node_type=node_type,
    index=torch.arange(num_nodes).long(),
)

print('===================================')
print(f'Number of nodes: {data.num_nodes}')
print(f'Number of edges: {num_edges}')
print(f'Number of node features: {data.x.shape[1]}')
print(f'Number of edge features: {data.edge_attr.shape[1]}')
print(f'Node storage dtype: {data.x.dtype}')
print(f'Edge storage dtype: {data.edge_attr.dtype}')
print(f'Node memmap path: {node_memmap_path}')
print(f'Number of labeled nodes: {int(data.has_target.sum())}')
for group, stats in feature_stats.items():
    print(f'Node type {group}: {len(stats["feature_idx"])} active features, model input dim = {2 * len(stats["feature_idx"])}')
print('===================================')


Streaming node table from parquet...


Node parquet batches:   0%|          | 0/67 [00:00<?, ?it/s]

Loading edge table...
Number of nodes: 3345036
Number of edges: 3907086
Number of node features: 244
Number of edge features: 44
Node storage dtype: torch.float16
Edge storage dtype: torch.float16
Node memmap path: /content/gnn-multitarget-2025/df_nodes.float16.mmap
Number of labeled nodes: 10046
Node type 0: 134 active features, model input dim = 268
Node type 1: 162 active features, model input dim = 324


## Training Helpers

CV считается по разметке соревнования, а обучение остается transductive: граф целиком доступен как контекст.


In [14]:
def make_loader(data: Data, input_nodes: torch.Tensor, config: dict, shuffle: bool):
    return NeighborLoader(
        data,
        input_nodes=input_nodes,
        batch_size=config['batch_size'],
        shuffle=shuffle,
        num_neighbors=config['num_neighbors'],
    )


def compute_pos_weights(y: torch.Tensor, label_mask: torch.Tensor, train_nodes: torch.Tensor) -> torch.Tensor:
    weights = []
    for task_idx in range(y.shape[1]):
        observed_mask = label_mask[train_nodes, task_idx]
        targets = y[train_nodes, task_idx][observed_mask]
        pos = float(targets.sum().item())
        neg = float(observed_mask.sum().item() - pos)
        weights.append(max(neg / max(pos, 1.0), 1.0))
    return torch.tensor(weights, dtype=torch.float32)


def score_predictions(df_score: pd.DataFrame, df_truth: pd.DataFrame, target_names: list[str]) -> dict:
    merged = df_truth.merge(df_score, on=['index', 'label_type'], how='inner')
    metrics = {'overall_auc': roc_auc_score(merged['y'], merged['score']), 'per_target_auc': {}}
    for target_name in target_names:
        target_df = merged[merged['label_type'] == target_name]
        try:
            metrics['per_target_auc'][target_name] = roc_auc_score(target_df['y'], target_df['score'])
        except ValueError:
            metrics['per_target_auc'][target_name] = np.nan
    return metrics


def format_metrics(metrics: dict) -> str:
    per_target = ', '.join(
        f"{name}={value:.4f}" if not np.isnan(value) else f"{name}=nan"
        for name, value in metrics['per_target_auc'].items()
    )
    return f"overall={metrics['overall_auc']:.4f} | {per_target}"


In [15]:
class EdgeAwareMultiTargetGNN(nn.Module):
    def __init__(self, feature_stats: dict, edge_dim: int, target_names: list[str], config: dict):
        super().__init__()
        self.target_names = target_names
        self.hidden_size = config['hidden_size']
        self.dropout_rate = config['dropout']
        self.group_ids = sorted(feature_stats.keys())
        self.input_blocks = nn.ModuleDict()

        for group in self.group_ids:
            stats = feature_stats[group]
            input_dim = 2 * len(stats['feature_idx'])
            self.input_blocks[str(group)] = nn.Sequential(
                nn.Linear(input_dim, config['input_mlp_size']),
                nn.LayerNorm(config['input_mlp_size']),
                nn.GELU(),
                nn.Dropout(self.dropout_rate),
                nn.Linear(config['input_mlp_size'], self.hidden_size),
                nn.LayerNorm(self.hidden_size),
                nn.GELU(),
            )
            self.register_buffer(f'group_{group}_feature_idx', torch.tensor(stats['feature_idx'], dtype=torch.long))
            self.register_buffer(f'group_{group}_median', torch.tensor(stats['median'], dtype=torch.float32))
            self.register_buffer(f'group_{group}_mean', torch.tensor(stats['mean'], dtype=torch.float32))
            self.register_buffer(f'group_{group}_std', torch.tensor(stats['std'], dtype=torch.float32))

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        for _ in range(3):
            self.convs.append(
                TransformerConv(
                    self.hidden_size,
                    self.hidden_size,
                    heads=config['heads'],
                    concat=False,
                    edge_dim=edge_dim,
                    beta=True,
                    dropout=self.dropout_rate,
                )
            )
            self.norms.append(nn.LayerNorm(self.hidden_size))
            self.dropouts.append(nn.Dropout(self.dropout_rate))

        head_input_dim = self.hidden_size * len(self.convs)
        self.heads = nn.ModuleDict(
            {
                name: nn.Sequential(
                    nn.Linear(head_input_dim, self.hidden_size),
                    nn.LayerNorm(self.hidden_size),
                    nn.GELU(),
                    nn.Dropout(self.dropout_rate),
                    nn.Linear(self.hidden_size, 1),
                )
                for name in self.target_names
            }
        )

    def encode_inputs(self, x: torch.Tensor, node_type: torch.Tensor) -> torch.Tensor:
        hidden = torch.zeros((x.size(0), self.hidden_size), device=x.device, dtype=torch.float32)
        for group in self.group_ids:
            group_mask = node_type == group
            if not bool(group_mask.any()):
                continue
            feature_idx = getattr(self, f'group_{group}_feature_idx')
            medians = getattr(self, f'group_{group}_median')
            means = getattr(self, f'group_{group}_mean')
            stds = getattr(self, f'group_{group}_std')

            x_group = x[group_mask][:, feature_idx].float()
            missing_mask = torch.isnan(x_group)
            x_filled = torch.where(missing_mask, medians.unsqueeze(0), x_group)
            x_norm = (x_filled - means.unsqueeze(0)) / stds.unsqueeze(0)
            group_input = torch.cat([x_norm, missing_mask.float()], dim=1)
            hidden[group_mask] = self.input_blocks[str(group)](group_input)
        return hidden

    def forward(self, data: Data) -> dict:
        x = self.encode_inputs(data.x, data.node_type.long())
        edge_attr = data.edge_attr.float()
        representations = []
        for conv, norm, dropout in zip(self.convs, self.norms, self.dropouts):
            x = conv(x, data.edge_index, edge_attr)
            x = norm(x)
            x = F.gelu(x)
            x = dropout(x)
            representations.append(x)
        graph_repr = torch.cat(representations, dim=1)
        return {name: head(graph_repr).squeeze(-1) for name, head in self.heads.items()}


In [16]:
def compute_batch_loss(batch: Data, logits: dict, pos_weights: torch.Tensor, target_names: list[str]) -> torch.Tensor:
    seed_size = batch.batch_size
    losses = []
    for task_idx, target_name in enumerate(target_names):
        task_mask = batch.label_mask[:seed_size, task_idx]
        if bool(task_mask.any()):
            task_logits = logits[target_name][:seed_size][task_mask]
            task_targets = batch.y[:seed_size, task_idx][task_mask]
            task_loss = F.binary_cross_entropy_with_logits(
                task_logits,
                task_targets,
                pos_weight=pos_weights[task_idx],
                reduction='mean',
            )
            losses.append(task_loss)
    if not losses:
        raise RuntimeError('Encountered a batch without labeled seed nodes.')
    return torch.stack(losses).mean()


def predict_scores(model: nn.Module, loader_batches, device: str, target_names: list[str]) -> pd.DataFrame:
    rows = []
    model.eval()
    with torch.no_grad():
        for batch in loader_batches:
            batch = batch.to(device)
            logits = model(batch)
            seed_size = batch.batch_size
            batch_scores = {
                name: torch.sigmoid(logits[name][:seed_size]).detach().cpu().numpy()
                for name in target_names
            }
            batch_scores['index'] = batch.index[:seed_size].detach().cpu().numpy()
            rows.append(pd.DataFrame(batch_scores))
    wide_df = pd.concat(rows, ignore_index=True)
    return wide_df.melt(id_vars=['index'], value_vars=target_names, var_name='label_type', value_name='score')


def train_one_epoch(model: nn.Module, train_loader, optimizer, pos_weights: torch.Tensor, config: dict) -> float:
    model.train()
    batch_losses = []
    for batch in tqdm.tqdm(train_loader, leave=False):
        batch = batch.to(config['device'])
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch)
        loss = compute_batch_loss(batch, logits, pos_weights, config['target_names'])
        if torch.isnan(loss):
            raise FloatingPointError('NaN loss detected during training.')
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        optimizer.step()
        batch_losses.append(loss.detach().item())
    return float(np.mean(batch_losses))


def train_one_fold(data: Data, train_nodes: torch.Tensor, valid_nodes: torch.Tensor, df_train: pd.DataFrame, feature_stats: dict, config: dict) -> dict:
    model = EdgeAwareMultiTargetGNN(feature_stats, data.edge_attr.shape[1], config['target_names'], config).to(config['device'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=config['scheduler_patience'],
    )
    pos_weights = compute_pos_weights(data.y, data.label_mask, train_nodes).to(config['device'])
    train_loader = make_loader(data, train_nodes, config, shuffle=True)
    valid_batches = [batch for batch in make_loader(data, valid_nodes, config, shuffle=False)]
    valid_truth = df_train[df_train['index'].isin(valid_nodes.cpu().numpy())].copy()

    history = []
    best_state = None
    best_metrics = None
    best_pred = None
    best_epoch = 0
    best_score = -np.inf
    patience_counter = 0

    for epoch in range(1, config['max_epochs'] + 1):
        start_time = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, pos_weights, config)
        valid_pred = predict_scores(model, valid_batches, config['device'], config['target_names'])
        metrics = score_predictions(valid_pred, valid_truth, config['target_names'])
        scheduler.step(metrics['overall_auc'])
        epoch_time = time.time() - start_time
        lr = optimizer.param_groups[0]['lr']
        history.append(
            {
                'epoch': epoch,
                'train_loss': train_loss,
                'overall_auc': metrics['overall_auc'],
                'lr': lr,
                'epoch_time': epoch_time,
                **{f'auc_{name}': value for name, value in metrics['per_target_auc'].items()},
            }
        )
        print(
            f"fold epoch={epoch:02d} loss={train_loss:.4f} lr={lr:.2e} time={epoch_time:.1f}s "
            f"{format_metrics(metrics)}"
        )

        if metrics['overall_auc'] > best_score + 1e-5:
            best_score = metrics['overall_auc']
            best_state = copy.deepcopy(model.state_dict())
            best_metrics = metrics
            best_pred = valid_pred.copy()
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config['early_stopping_patience']:
                print(f'Early stopping at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    return {
        'model': model,
        'best_epoch': best_epoch,
        'best_metrics': best_metrics,
        'best_pred': best_pred,
        'history': pd.DataFrame(history),
    }


def fit_full_model(data: Data, train_nodes: torch.Tensor, feature_stats: dict, config: dict, num_epochs: int) -> nn.Module:
    model = EdgeAwareMultiTargetGNN(feature_stats, data.edge_attr.shape[1], config['target_names'], config).to(config['device'])
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    pos_weights = compute_pos_weights(data.y, data.label_mask, train_nodes).to(config['device'])
    train_loader = make_loader(data, train_nodes, config, shuffle=True)

    for epoch in range(1, num_epochs + 1):
        start_time = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, pos_weights, config)
        epoch_time = time.time() - start_time
        print(f'final epoch={epoch:02d} loss={train_loss:.4f} time={epoch_time:.1f}s')

    return model


## Cross-Validation

CV служит для честной offline-оценки и выбора числа эпох для финального full-fit.


In [17]:
folds, labeled_nodes = make_split_folds(
    df_y=df_y,
    node_type=data.node_type,
    target_names=CONFIG['target_names'],
    n_splits=CONFIG['num_folds'],
    seed=CONFIG['seed'],
)

print('Number of folds:', len(folds))
print('Number of labeled nodes:', len(labeled_nodes))
print('Number of test nodes:', len(test_index))


Number of folds: 3
Number of labeled nodes: 10046
Number of test nodes: 4949


In [11]:
fold_results = []
oof_predictions = []
best_epochs = []

if CONFIG['run_cv']:
    for fold in folds:
        print('=' * 80)
        print(f"Fold {fold['fold_id']}")
        fold_result = train_one_fold(
            data=data,
            train_nodes=fold['train_nodes'],
            valid_nodes=fold['valid_nodes'],
            df_train=df_train,
            feature_stats=feature_stats,
            config=CONFIG,
        )
        fold_metrics = fold_result['best_metrics']
        print(f"Best epoch: {fold_result['best_epoch']}")
        print('Best metrics:', format_metrics(fold_metrics))

        fold_results.append(
            {
                'fold_id': fold['fold_id'],
                'best_epoch': fold_result['best_epoch'],
                'overall_auc': fold_metrics['overall_auc'],
                **fold_metrics['per_target_auc'],
            }
        )
        oof_predictions.append(fold_result['best_pred'])
        best_epochs.append(fold_result['best_epoch'])

        del fold_result['model']
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if fold_results:
    cv_results_df = pd.DataFrame(fold_results)
    print(cv_results_df)
    oof_predictions_df = pd.concat(oof_predictions, ignore_index=True)
    oof_metrics = score_predictions(oof_predictions_df, df_train, CONFIG['target_names'])
    print('OOF metrics:', format_metrics(oof_metrics))
    if CONFIG['save_oof']:
        oof_predictions_df.to_csv('oof_predictions.csv', index=False)
        print('Saved oof_predictions.csv')
else:
    cv_results_df = pd.DataFrame()
    oof_predictions_df = None
    oof_metrics = None


Fold 0


  0%|          | 0/27 [00:00<?, ?it/s]

fold epoch=01 loss=1.1631 lr=2.00e-03 time=15.4s overall=0.7160 | label_3=0.6675, label_4=0.6236, label_5=0.7168, label_6=0.7280


  0%|          | 0/27 [00:00<?, ?it/s]

fold epoch=02 loss=1.0402 lr=2.00e-03 time=13.7s overall=0.6793 | label_3=0.6478, label_4=0.6354, label_5=0.7064, label_6=0.7305


...

fold epoch=04 loss=0.9169 lr=1.00e-03 time=15.4s overall=0.6924 | label_3=0.5195, label_4=0.6374, label_5=0.6946, label_6=0.6949


  0%|          | 0/27 [00:00<?, ?it/s]

fold epoch=05 loss=0.8627 lr=1.00e-03 time=15.4s overall=0.7072 | label_3=0.5867, label_4=0.6433, label_5=0.7178, label_6=0.7194


  0%|          | 0/27 [00:00<?, ?it/s]

fold epoch=06 loss=0.7433 lr=1.00e-03 time=15.4s overall=0.7088 | label_3=0.5979, label_4=0.6446, label_5=0.7101, label_6=0.7106
Early stopping at epoch 6
Best epoch: 1
Best metrics: overall=0.7137 | label_3=0.5728, label_4=0.6403, label_5=0.7118, label_6=0.7505
   fold_id  best_epoch  overall_auc   label_3   label_4   label_5   label_6
0        0           1     0.716002  0.667510  0.623625  0.716762  0.727990
1        1           7     0.700849  0.582722  0.575187  0.712890  0.748208
2        2           1     0.713726  0.572821  0.640325  0.711778  0.750525
OOF metrics: overall=0.7018 | label_3=0.5881, label_4=0.5819, label_5=0.7039, label_6=0.7332
Saved oof_predictions.csv


## Final Fit And Submission

Финальная модель учится на всех размеченных вершинах число эпох, равное среднему лучшему epoch по CV.


In [12]:
if CONFIG['run_final_fit']:
    if best_epochs:
        final_epochs = int(np.clip(round(float(np.mean(best_epochs))), 1, CONFIG['max_epochs']))
    else:
        final_epochs = max(1, CONFIG['max_epochs'] // 2)

    print('Final training epochs:', final_epochs)
    final_model = fit_full_model(
        data=data,
        train_nodes=labeled_nodes,
        feature_stats=feature_stats,
        config=CONFIG,
        num_epochs=final_epochs,
    )
    test_batches = [batch for batch in make_loader(data, test_index, CONFIG, shuffle=False)]
    df_test_score = predict_scores(final_model, test_batches, CONFIG['device'], CONFIG['target_names'])
    submission = df_test.merge(df_test_score, on=['index', 'label_type'], how='left')
    assert submission['score'].notnull().all(), 'Missing scores in submission.'
    assert np.isfinite(submission['score']).all(), 'Non-finite scores in submission.'
    assert ((submission['score'] >= 0.0) & (submission['score'] <= 1.0)).all(), 'Scores must be probabilities.'
    submission[['task_id', 'score']].to_csv('submission.csv', index=False)
    print(submission[['task_id', 'score']].head())
    print('Saved submission.csv')
else:
    final_model = None
    submission = None


Final training epochs: 3


  0%|          | 0/40 [00:00<?, ?it/s]

final epoch=01 loss=1.1364 time=19.7s


  0%|          | 0/40 [00:00<?, ?it/s]

final epoch=02 loss=1.0539 time=19.8s


  0%|          | 0/40 [00:00<?, ?it/s]

final epoch=03 loss=1.0068 time=19.8s
   task_id     score
0        0  0.443348
1        1  0.269151
2        6  0.465309
3        7  0.496943
4        8  0.422385
Saved submission.csv


In [23]:
from pathlib import Path
import getpass
import os

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
access_token_path = kaggle_dir / 'access_token'

if not access_token_path.exists() or access_token_path.read_text().strip() in {'', 'ТВОЙ_KAGGLE_ACCESS_TOKEN'}:
    token = getpass.getpass('Paste Kaggle access token: ').strip()
    if not token:
        raise ValueError('Kaggle access token is empty.')
    access_token_path.write_text(token)
    access_token_path.chmod(0o600)

os.environ['KAGGLE_API_TOKEN'] = access_token_path.read_text().strip()
print(f'Kaggle token is configured at {access_token_path}')


Kaggle token is configured at /root/.kaggle/access_token


In [24]:
import subprocess
import sys

subprocess.check_call([
    'kaggle', 'competitions', 'submit',
    '-c', 'gnn-multitarget-2025',
    '-f', 'submission.csv',
    '-m', 'edge-aware single-model baseline upgrade',
])

subprocess.check_call([
    'kaggle', 'competitions', 'submissions',
    '-c', 'gnn-multitarget-2025',
])


0

In [27]:
import subprocess

result = subprocess.run(
    ['kaggle', 'competitions', 'submissions', '-c', 'gnn-multitarget-2025'],
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)


fileName        date                        description                               status                     publicScore  privateScore  
--------------  --------------------------  ----------------------------------------  -------------------------  -----------  ------------  
submission.csv  2026-03-19 07:30:20.130000  edge-aware single-model baseline upgrade  SubmissionStatus.COMPLETE  0.73494      0.73249       


